# ROBERT R=100 emission and transmission validation

This notebook is the first end-to-end check for a new ROBERT installation. It performs the same four-stage experiment for thermal emission and transmission:

1. evaluate a known forward model with ROBERT's bundled R=100 H2O opacity;
2. add reproducible Gaussian noise with 60 ppm error bars to create 18 synthetic observations from 1.10 to 1.70 microns;
3. retrieve the injected H2O abundance with MultiNest; and
4. verify that the injected value lies inside the retrieved 95% credible interval and that the spectral fit has a reasonable reduced chi-square.

The cases are deliberately identifiable. Emission uses a fixed non-isothermal temperature profile, while transmission fixes the reference radius. Each retrieval therefore has one unknown, `log_H2O`, and tests the software stack without introducing an abundance-temperature or abundance-radius degeneracy.

Run every cell in order and watch the MultiNest output in this notebook. Use two CPU cores on a laptop or up to four in an interactive cluster allocation. Do not submit this short validation to a queue: live output makes missing libraries, MPI problems, opacity failures, and non-convergence immediately visible.

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display

def find_repo_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for directory in (candidate, *candidate.parents):
        if (directory / 'pyproject.toml').is_file() and (directory / 'src').is_dir():
            return directory
    raise RuntimeError('Could not find the ROBERT repository root')

ROOT = find_repo_root()
os.chdir(ROOT)
CONFIGS = {
    'emission': ROOT / 'configurations' / 'quickstart.yaml',
    'transmission': ROOT / 'configurations' / 'transmission.yaml',
}
WORKFLOW = ROOT / 'examples' / 'r100_injection_recovery.py'
CORES = 2  # Set this to 3 or 4 for a larger laptop or interactive cluster allocation.
if CORES not in (2, 3, 4):
    raise ValueError('Use 2, 3, or 4 cores for this validation')
MPIEXEC = shutil.which('mpiexec') or shutil.which('mpirun')
if MPIEXEC is None:
    raise RuntimeError('mpiexec/mpirun is unavailable; create the complete ROBERT Conda environment')
print(f'Repository: {ROOT}')
print(f'Python:     {sys.executable}')
print(f'MPI:        {MPIEXEC}')
print(f'Cores:      {CORES}')

## Live subprocess helper

Forward generation and retrieval run as ordinary foreground subprocesses. Their combined output is streamed line by line. Interrupting the notebook interrupts the active command; no detached or queued job is created.

In [ ]:
def run_live(command: list[str], label: str) -> float:
    print(f'\n{"=" * 88}\n{label}\n{"=" * 88}', flush=True)
    print(' '.join(command), flush=True)
    started = time.monotonic()
    process = subprocess.Popen(
        command,
        cwd=ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    )
    try:
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='', flush=True)
        returncode = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait(timeout=10)
        raise
    elapsed = time.monotonic() - started
    if returncode != 0:
        raise RuntimeError(f'{label} failed with exit code {returncode}')
    print(f'{label} completed in {elapsed:.1f} s', flush=True)
    return elapsed

## 1. Preflight MultiNest and both configurations

This confirms that Python can load PyMultiNest and its compiled MultiNest library before any scientific work starts. It then resolves both YAML files and confirms that the bundled R=100 opacity path is selected.

In [ ]:
run_live(
    [sys.executable, '-c', 'import mpi4py, pymultinest; print("MPI and MultiNest imports succeeded")'],
    'MultiNest import preflight',
)
for name, config in CONFIGS.items():
    run_live(
        [sys.executable, '-u', 'run_retrieval.py', '--config', str(config), '--validate-only'],
        f'{name.capitalize()} configuration preflight',
    )

## 2. Generate each forward truth and synthetic observation

For each case, ROBERT first writes `forward_model.npz` at the injected H2O abundance. That saved forward product—not a separate approximation—is then converted into an observation with independent, seeded Gaussian noise and 60 ppm uncertainties. The exact truth, random seed, input/output checksums, wavelength range, and file names are saved in `injection_truth.json`.

In [ ]:
for name, config in CONFIGS.items():
    run_live(
        [sys.executable, '-u', str(WORKFLOW), '--config', str(config), '--generate'],
        f'Generate {name} forward model and synthetic observation',
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for axis, (name, config) in zip(axes, CONFIGS.items(), strict=True):
    output = ROOT / 'examples' / 'outputs' / 'r100_validation' / name
    with np.load(output / 'forward_model.npz', allow_pickle=False) as forward:
        dataset = f'synthetic_{name}'
        wavelength = forward[f'{dataset}_wavelength_micron']
        truth_model = forward[f'{dataset}_model']
    with np.load(output / 'synthetic_observation.npz', allow_pickle=False) as observation:
        observed = observation['data']
        uncertainty = observation['err']
    axis.plot(wavelength, 1e6 * truth_model, color='black', label='Forward truth')
    axis.errorbar(wavelength, 1e6 * observed, yerr=1e6 * uncertainty, fmt='o', ms=4, label='Synthetic data')
    axis.set(title=name.capitalize(), xlabel='Wavelength (micron)', ylabel='Depth (ppm)')
    axis.legend()
fig.tight_layout()

## 3. Run and monitor the local MultiNest retrievals

Each command stays attached to this notebook and uses the selected two to four MPI ranks. Normal output includes the number of live points, acceptance rate, likelihood evaluations, evidence updates, and the final `Sampling finished` message. The configured run also writes `sampler_status.json` and an attempt journal under each `multinest/` directory.

A first run may spend a few seconds compiling numerical kernels and building Matplotlib's font cache. Sampling itself is intentionally short. If a command stops before `Sampling finished`, inspect the visible traceback before continuing.

In [ ]:
retrieval_seconds = {}
for name, config in CONFIGS.items():
    command = [
        MPIEXEC, '-n', str(CORES), sys.executable, '-u',
        'run_retrieval.py', '--config', str(config),
    ]
    retrieval_seconds[name] = run_live(command, f'Run and monitor {name} MultiNest retrieval')
retrieval_seconds

## 4. Verify recovery of the known inputs

The evaluator returns a non-zero exit status if either retrieval did not converge, if the injected abundance falls outside its 95% posterior credible interval, or if the reduced chi-square falls outside 0.35–1.90. The chi-square interval is deliberately wider than a production goodness-of-fit threshold because this compact example has only 17 residual degrees of freedom.

In [ ]:
for name, config in CONFIGS.items():
    run_live(
        [sys.executable, '-u', str(WORKFLOW), '--config', str(config), '--evaluate'],
        f'Compare {name} posterior with injected truth',
    )

In [ ]:
reports = {}
for name in CONFIGS:
    path = ROOT / 'examples' / 'outputs' / 'r100_validation' / name / 'injection_recovery_report.json'
    reports[name] = json.loads(path.read_text())

print(f'{"Case":14s} {"Pass":5s} {"Truth":>8s} {"Median":>8s} {"95% interval":>23s} {"chi2_red":>9s} {"ESS":>7s}')
print('-' * 84)
for name, report in reports.items():
    recovery = report['parameter_recoveries']['log_H2O']
    interval = f"[{recovery['q02_5']:.3f}, {recovery['q97_5']:.3f}]"
    print(
        f"{name:14s} {str(report['passed']):5s} {recovery['truth']:8.3f} "
        f"{recovery['median']:8.3f} {interval:>23s} "
        f"{report['reduced_chi_square']:9.3f} {report['effective_sample_size']:7.1f}"
    )
assert all(report['passed'] for report in reports.values())
print('\nPASS: emission and transmission both recovered their injected H2O abundance.')

In [ ]:
colours = {'emission': '#c44e52', 'transmission': '#4c72b0'}
for name in CONFIGS:
    plot_dir = ROOT / 'examples' / 'outputs' / 'r100_validation' / name / 'plots' / 'multinest'
    fit_path = plot_dir / 'fit_spectrum_residuals.png'
    print(f'\n{name.upper()} posterior-predictive fit')
    if fit_path.is_file():
        display(Image(filename=str(fit_path)))
    else:
        print(f'Missing expected plot: {fit_path}')

    result_path = ROOT / 'examples' / 'outputs' / 'r100_validation' / name / 'multinest' / 'result_arrays.npz'
    with np.load(result_path, allow_pickle=False) as archive:
        samples = np.asarray(archive['samples'], dtype=float)[:, 0]
        weights = np.asarray(archive['weights'], dtype=float)
    weights /= weights.sum()
    recovery = reports[name]['parameter_recoveries']['log_H2O']
    figure, axis = plt.subplots(figsize=(7.2, 4.2))
    axis.hist(samples, bins=35, weights=weights, density=True, color=colours[name], alpha=0.72)
    axis.axvspan(recovery['q02_5'], recovery['q97_5'], color=colours[name], alpha=0.14, label='95% credible interval')
    axis.axvline(recovery['median'], color=colours[name], linewidth=2, label=f"Posterior median: {recovery['median']:.3f}")
    axis.axvline(recovery['truth'], color='black', linestyle='--', linewidth=2, label=f"Injected truth: {recovery['truth']:.3f}")
    axis.set(title=f'{name.capitalize()} H2O injection recovery', xlabel='log10 H2O VMR', ylabel='Posterior density')
    axis.set_yticks([])
    axis.legend()
    figure.tight_layout()
    plt.show()

## Interpreting the result

A final `PASS` means the complete local path worked: package data discovery, exo_k preparation, emission and transmission radiative transfer, synthetic-data serialization, MPI, the compiled MultiNest library, posterior serialization, plotting, and recovery evaluation.

This is an installation and workflow validation, not evidence that R=100 is sufficient for every scientific dataset or that a one-molecule atmosphere is realistic. Once both checks pass, copy a science configuration, introduce the required molecules and physical parameters, and establish opacity-resolution and model convergence for the actual observations.